# Track designer

Draw a track as a rough polygon, smooth it into a drivable loop, preview it,
and install it — after `install_track` the track works everywhere a track
name is accepted (experiments, `rollout_video`, the CLI).

The same machinery imports any of the 126 official DeepRacer tracks:
`fetch_official_track("Vegas_track")`.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from deepracer_genesis.tools.track_builder import (
    stadium, route_from_waypoints, build_route, build_track_mesh,
    plot_track, plot_wireframe, install_track, fetch_official_track,
    track_metrics,
)

## 1. Define the centerline

Sketch a centerline in meters and pick a track `width`; the loop closes
automatically and your waypoints ARE the track. `stadium(straight, radius)`
generates a pill / stretched-oval end-to-end; `route_from_waypoints` densifies
the polyline and lays the inner/outer borders (`n_waypoints` sets resolution —
bump it so curved ends stay smooth).

Sizing for a room: the paved footprint is
`(straight + 2*radius + width) x (2*radius + width)`, so keep both within the
floor. `radius` is the centerline turn radius and must exceed `width / 2` or
the inner border self-pinches. Official tracks are ~1.06 m wide; the car is ~0.2 m.

In [ ]:
# Oval sized for a 1.5 x 1.6 m room with a ~10 cm grass margin all around:
# footprint 1.4 x 1.3 m. The room is nearly square, so `straight` is tiny and
# the pill is effectively an oval. Paved footprint is
# (straight + 2*radius + width) x (2*radius + width); keep 2*radius + width
# <= room_short_side - 2*margin.
room = (1.6, 1.5)                                 # (long, short) side in meters
waypoints = stadium(straight=0.10, radius=0.40)
width = 0.5                                       # road width in meters

route = route_from_waypoints(waypoints, width, n_waypoints=160)  # smooth arcs

# prefer sketching rough corners and letting the tool round them?
# route = build_route(waypoints, half_width=width / 2, smooth_passes=3)

# clean printable layout: grass to the walls, thick white edge strips,
# meter-scale dashed centerline, no axes
fig, _ = plot_track(route, room=room, border_strip=0.06,
                    dash_len=0.12, dash_gap=0.14, centerline_width=0.025,
                    out_path="donut_track_print.png")
fig.savefig("donut_track_print.pdf", facecolor=fig.get_facecolor(),
            bbox_inches="tight", pad_inches=0)   # vector, for scaled printing

# wireframe: borders, waypoints, start marker + driving direction, room bounds
plot_wireframe(route, waypoints, room=room, out_path="donut_track_wireframe.png")

metrics = track_metrics(route)
print(metrics)   # everything you need to lay the track out physically

## 2. Install it

Writes `route.npy` + a generated road mesh (asphalt / border lines / dashed
centerline as tiny solid textures — renders identically under Madrona, Nyx
and the rasterizer) under `assets/tracks/generated/<name>/` and registers
the name. Generated tracks are re-discovered automatically in every process.

In [3]:
install_track("donut_track", route)
from deepracer_genesis.envs.track import TRACKS
print(sorted(TRACKS))

[I 07/31/26 21:04:23.283 217847] [shell.py:_shell_pop_print@25] Graphical python shell detected, using wrapped sys.stdout


['2022_reinvent_champ', 'AWS_track', 'Austin', 'Bowtie_track', 'Canada_Training', 'China_track', 'Mexico_track', 'Monaco', 'New_York_Track', 'Oval_track', 'Singapore', 'Spain_track', 'Straight_track', 'Tokyo_Training_track', 'Vegas_track', 'donut_track', 'reInvent2019_track', 'reinvent_base']


## 3. Sanity-drive it

A privileged P-controller drives 4 cars for 10 s and renders the bird's-eye
view. If the cars lap without leaving the road, the track is well-formed
(no self-intersections, radii the car can steer).

In [4]:
import torch, genesis as gs
import imageio.v2 as imageio
gs.init(backend=gs.cuda, logging_level="warning")
from deepracer_genesis.configs.cfgs import get_env_cfg
from deepracer_genesis.envs import DeepRacerEnv

cfg = get_env_cfg(vision=False, track="donut_track")
cfg.update(spectator=True, spectator_res=(960, 720))
env = DeepRacerEnv(num_envs=4, env_cfg=cfg)
prog = torch.zeros(4, device=env.device)
for _ in range(300):
    lat = env.lateral * env.dir_sign / env.half_width.clamp(min=0.1)
    steer = (-(1.1 * lat + 0.9 * torch.sin(env.heading_err))).clamp(-1, 1)
    env.step(torch.stack([steer, torch.full_like(steer, -0.3)], dim=1))
    prog += env.d_progress
imageio.imwrite("/tmp/donut_track_topdown.png", env.render_spectator())
print("mean progress:", round(prog.mean().item(), 1), "m in 10 s |",
      "offtrack events:", int(env.offtrack_buf.sum().item()))
del env    # scenes CAN be rebuilt in-process — re-run after editing the track

[Genesis] [21:04:26] [WARNING] Mesh is not watertight. Falling back to convex hull for estimating inertial properties.
[Genesis] [21:04:26] [WARNING] Mesh is not watertight. Falling back to convex hull for estimating inertial properties.
[Genesis] [21:04:26] [WARNING] Mesh is not watertight. Falling back to convex hull for estimating inertial properties.
[Genesis] [21:04:26] [WARNING] Mesh is not watertight. Falling back to convex hull for estimating inertial properties.
[Genesis] [21:04:26] [WARNING] Mesh is not watertight. Falling back to convex hull for estimating inertial properties.
[Genesis] [21:04:26] [WARNING] Mesh is not watertight. Falling back to convex hull for estimating inertial properties.
[Genesis] [21:04:26] [WARNING] Mesh is not watertight. Falling back to convex hull for estimating inertial properties.
[Genesis] [21:04:27] [WARNING] This property is deprecated and will be removed in future release. Please use 'dofs_idx_local' instead.
[Genesis] [21:04:27] [WARNING] T

AssertionError: spectator camera not enabled (cfg['spectator'])

In [ ]:
from IPython.display import Image
Image("/tmp/donut_track_topdown.png", width=720)

## 4. Train on it

Any experiment takes the new name — single-file style:

```python
from deepracer_genesis.experiment import Experiment, FeatureEnvironment, VectorPolicy, run

class MyTrackRacer(Experiment):
    total_env_steps = 5_000_000
    eval_every_steps = 1_000_000
    def pipeline(self):
        return FeatureEnvironment(num_envs=1024, tracks=("donut_track",)) >> VectorPolicy()

run(MyTrackRacer)
```

...or watch a policy trained elsewhere drive it:
`rollout_video("feature_baseline", track="donut_track")`.

In [ ]:
from deepracer_genesis import tracks

tracks.names()